# Unified V2 Results — All Datasets
Yelp (classification) · GSM8K (math reasoning) · Alpaca (instruction following)

Methods: homo_r8, hetero_pad, flexlora, hetlora, spa_m  
Heterogeneity: α ∈ {0.5, 0.1}  
Seeds: 42, 43, 44 (gen tasks) / 42–46 (Yelp)

In [ ]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── paths ──────────────────────────────────────────────────────────────────────
# Change BASE to wherever results_v2 lives on this machine
BASE = os.path.expanduser('/home/sp2ai/FedLLM-Re/rework/results_v2')
DATASET_DIRS = {
    'yelp':   os.path.join(BASE, 'yelp'),
    'gsm8k':  os.path.join(BASE, 'gsm8k'),
    'alpaca': os.path.join(BASE, 'alpaca'),
}
PRIMARY_METRIC = {
    'yelp':   'accuracy',
    'gsm8k':  'exact_match',
    'alpaca': 'rouge_l',
}
METRIC_LABEL = {
    'yelp':   'Accuracy',
    'gsm8k':  'Exact Match',
    'alpaca': 'ROUGE-L',
}
DATASET_NAME = {
    'yelp':   'Yelp Review (Classification)',
    'gsm8k':  'GSM8K (Math Reasoning)',
    'alpaca': 'Alpaca (Instruction Following)',
}

# ── method display config ──────────────────────────────────────────────────────
# Main paper methods (ordered as in Table 1)
PAPER_METHODS = ['homo_r8', 'hetero_pad', 'flexlora', 'hetlora', 'spa_m']
ALL_METHODS   = ['homo_r4', 'homo_r8', 'hetero_pad', 'flexlora', 'hetero_spa', 'hetlora', 'spa_m']

LABELS = {
    'homo_r4':    'Homo r=4',
    'homo_r8':    'Homo r=8',
    'hetero_pad': 'Hetero-Pad',
    'flexlora':   'FlexLoRA',
    'hetero_spa': 'Hetero-SPA',
    'hetlora':    'HetLoRA',
    'spa_m':      'SPA-M (ours)',
}
COLORS = {
    'homo_r4':    '#aaaaaa',
    'homo_r8':    '#888888',
    'hetero_pad': '#4e79a7',
    'flexlora':   '#f28e2b',
    'hetero_spa': '#76b7b2',
    'hetlora':    '#e15759',
    'spa_m':      '#59a14f',
}
LINESTYLES = {
    'homo_r4':    ':',
    'homo_r8':    '--',
    'hetero_pad': '-.',
    'flexlora':   '-',
    'hetero_spa': '-',
    'hetlora':    '-',
    'spa_m':      '-',
}
LINEWIDTHS = {
    'homo_r4':    1.2,
    'homo_r8':    1.2,
    'hetero_pad': 1.5,
    'flexlora':   1.5,
    'hetero_spa': 1.5,
    'hetlora':    1.8,
    'spa_m':      2.2,
}

print('Config loaded.')
for ds, path in DATASET_DIRS.items():
    n = len(glob.glob(os.path.join(path, '*.json')))
    print(f'  {ds}: {n} files in {path}')

In [ ]:
# ── data loading ───────────────────────────────────────────────────────────────

def load_dataset(dataset: str) -> pd.DataFrame:
    """Load all JSON result files for a dataset into a tidy DataFrame."""
    path = DATASET_DIRS[dataset]
    metric = PRIMARY_METRIC[dataset]
    rows = []
    seen = set()  # (method, alpha, seed) dedup
    for fp in sorted(glob.glob(os.path.join(path, '*.json'))):
        try:
            with open(fp) as f:
                data = json.load(f)
        except Exception as e:
            print(f'  WARNING: could not read {fp}: {e}')
            continue
        method = data.get('method', 'unknown')
        seed   = data.get('seed', -1)
        alpha  = data.get('alpha', -1)
        key    = (method, alpha, seed)
        if key in seen:
            continue
        seen.add(key)
        for r in data.get('rounds', []):
            val = r.get(metric)
            if val is None:
                continue
            rows.append({
                'dataset': dataset,
                'method':  method,
                'alpha':   alpha,
                'seed':    seed,
                'round':   r['round'],
                'metric':  float(val),
                'loss':    r.get('avg_loss', np.nan),
            })
    return pd.DataFrame(rows)


def summary_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Per (method, alpha): Mean-L5, Best, std across seeds."""
    rows = []
    for (method, alpha), g in df.groupby(['method', 'alpha']):
        per_seed = []
        for seed, sg in g.groupby('seed'):
            vals = sg.sort_values('round')['metric'].values
            if len(vals) == 0:
                continue
            mean_l5 = float(np.mean(vals[-5:])) if len(vals) >= 5 else float(np.mean(vals))
            best    = float(np.max(vals))
            per_seed.append({'mean_l5': mean_l5, 'best': best})
        if not per_seed:
            continue
        ml5 = [p['mean_l5'] for p in per_seed]
        bst = [p['best']    for p in per_seed]
        rows.append({
            'method':       method,
            'alpha':        alpha,
            'mean_l5':      np.mean(ml5),
            'mean_l5_std':  np.std(ml5),
            'best':         np.mean(bst),
            'best_std':     np.std(bst),
            'n_seeds':      len(per_seed),
        })
    return pd.DataFrame(rows)


def mean_curve(df: pd.DataFrame, method: str, alpha: float) -> tuple:
    """Returns (rounds_array, mean_metric, std_metric) averaged across seeds."""
    sub = df[(df['method'] == method) & (df['alpha'] == alpha)]
    if sub.empty:
        return None, None, None
    pivot = sub.pivot_table(index='round', columns='seed', values='metric', aggfunc='mean')
    rounds = pivot.index.values
    mu  = pivot.mean(axis=1).values
    std = pivot.std(axis=1).values
    return rounds, mu, std


print('Functions defined.')

In [ ]:
# ── load all datasets ──────────────────────────────────────────────────────────
dfs = {ds: load_dataset(ds) for ds in ['yelp', 'gsm8k', 'alpaca']}
stats = {ds: summary_stats(dfs[ds]) for ds in dfs}

for ds, df in dfs.items():
    if df.empty:
        print(f'{ds}: NO DATA')
        continue
    methods_found = df['method'].unique().tolist()
    alphas_found  = sorted(df['alpha'].unique().tolist())
    seeds_found   = sorted(df['seed'].unique().tolist())
    print(f'{ds}: {len(df)} rows | methods={methods_found} | alphas={alphas_found} | seeds={seeds_found}')

---
## 1  Yelp Review — Classification

In [ ]:
# ── Yelp convergence curves ────────────────────────────────────────────────────
ds = 'yelp'
df = dfs[ds]
if df.empty:
    print('No Yelp data.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    for ax, alpha in zip(axes, [0.5, 0.1]):
        for m in PAPER_METHODS:
            rounds, mu, std = mean_curve(df, m, alpha)
            if rounds is None:
                continue
            ax.plot(rounds, mu,
                    label=LABELS[m], color=COLORS[m],
                    ls=LINESTYLES[m], lw=LINEWIDTHS[m])
            ax.fill_between(rounds, mu - std, mu + std,
                            alpha=0.12, color=COLORS[m])
        ax.set_title(f'Yelp  α={alpha}', fontweight='bold')
        ax.set_xlabel('Round')
        ax.set_ylabel(METRIC_LABEL[ds])
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig('../figures/yelp_convergence_v2.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Yelp summary table ─────────────────────────────────────────────────────────
ds = 'yelp'
st = stats[ds]
if st.empty:
    print('No Yelp stats.')
else:
    rows = []
    for m in PAPER_METHODS:
        row = {'Method': LABELS.get(m, m)}
        for alpha in [0.5, 0.1]:
            sub = st[(st['method'] == m) & (st['alpha'] == alpha)]
            if sub.empty:
                row[f'MeanL5 α={alpha}'] = '—'
                row[f'Best α={alpha}']   = '—'
            else:
                r = sub.iloc[0]
                row[f'MeanL5 α={alpha}'] = f"{r['mean_l5']*100:.2f} ±{r['mean_l5_std']*100:.2f}"
                row[f'Best α={alpha}']   = f"{r['best']*100:.2f}"
        rows.append(row)
    print('Yelp Results (%, Mean of Last 5 Rounds ± std across seeds)')
    display(pd.DataFrame(rows).set_index('Method'))

---
## 2  GSM8K — Math Reasoning

In [ ]:
# ── GSM8K convergence curves ───────────────────────────────────────────────────
ds = 'gsm8k'
df = dfs[ds]
if df.empty:
    print('No GSM8K data.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    for ax, alpha in zip(axes, [0.5, 0.1]):
        for m in PAPER_METHODS:
            rounds, mu, std = mean_curve(df, m, alpha)
            if rounds is None:
                continue
            ax.plot(rounds, mu,
                    label=LABELS[m], color=COLORS[m],
                    ls=LINESTYLES[m], lw=LINEWIDTHS[m])
            ax.fill_between(rounds, mu - std, mu + std,
                            alpha=0.12, color=COLORS[m])
        ax.set_title(f'GSM8K  α={alpha}', fontweight='bold')
        ax.set_xlabel('Round')
        ax.set_ylabel(METRIC_LABEL[ds])
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig('../figures/gsm8k_convergence_v2.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── GSM8K summary table ────────────────────────────────────────────────────────
ds = 'gsm8k'
st = stats[ds]
if st.empty:
    print('No GSM8K stats.')
else:
    rows = []
    for m in PAPER_METHODS:
        row = {'Method': LABELS.get(m, m)}
        for alpha in [0.5, 0.1]:
            sub = st[(st['method'] == m) & (st['alpha'] == alpha)]
            if sub.empty:
                row[f'MeanL5 α={alpha}'] = '—'
                row[f'Best α={alpha}']   = '—'
            else:
                r = sub.iloc[0]
                row[f'MeanL5 α={alpha}'] = f"{r['mean_l5']*100:.2f} ±{r['mean_l5_std']*100:.2f}"
                row[f'Best α={alpha}']   = f"{r['best']*100:.2f}"
        rows.append(row)
    print('GSM8K Results (%, Mean of Last 5 Rounds ± std across seeds)')
    display(pd.DataFrame(rows).set_index('Method'))

---
## 3  Alpaca — Instruction Following

In [ ]:
# ── Alpaca convergence curves ──────────────────────────────────────────────────
ds = 'alpaca'
df = dfs[ds]
if df.empty:
    print('No Alpaca data.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    for ax, alpha in zip(axes, [0.5, 0.1]):
        for m in PAPER_METHODS:
            rounds, mu, std = mean_curve(df, m, alpha)
            if rounds is None:
                continue
            ax.plot(rounds, mu,
                    label=LABELS[m], color=COLORS[m],
                    ls=LINESTYLES[m], lw=LINEWIDTHS[m])
            ax.fill_between(rounds, mu - std, mu + std,
                            alpha=0.12, color=COLORS[m])
        ax.set_title(f'Alpaca  α={alpha}', fontweight='bold')
        ax.set_xlabel('Round')
        ax.set_ylabel(METRIC_LABEL[ds])
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig('../figures/alpaca_convergence_v2.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Alpaca summary table ───────────────────────────────────────────────────────
ds = 'alpaca'
st = stats[ds]
if st.empty:
    print('No Alpaca stats.')
else:
    rows = []
    for m in PAPER_METHODS:
        row = {'Method': LABELS.get(m, m)}
        for alpha in [0.5, 0.1]:
            sub = st[(st['method'] == m) & (st['alpha'] == alpha)]
            if sub.empty:
                row[f'MeanL5 α={alpha}'] = '—'
                row[f'Best α={alpha}']   = '—'
            else:
                r = sub.iloc[0]
                row[f'MeanL5 α={alpha}'] = f"{r['mean_l5']:.4f} ±{r['mean_l5_std']:.4f}"
                row[f'Best α={alpha}']   = f"{r['best']:.4f}"
        rows.append(row)
    print('Alpaca Results (ROUGE-L, Mean of Last 5 Rounds ± std across seeds)')
    display(pd.DataFrame(rows).set_index('Method'))

---
## 4  Combined Bar Chart — All Datasets × Both Alphas

In [ ]:
# ── Combined grouped bar chart ─────────────────────────────────────────────────
# One row per dataset, columns = methods, two alpha panels

DATASETS = ['yelp', 'gsm8k', 'alpaca']
ALPHAS   = [0.5, 0.1]

fig, axes = plt.subplots(len(DATASETS), len(ALPHAS),
                          figsize=(12, 9), sharey=False)

for row_i, ds in enumerate(DATASETS):
    st = stats[ds]
    is_pct = ds in ('yelp', 'gsm8k')
    for col_j, alpha in enumerate(ALPHAS):
        ax = axes[row_i][col_j]
        vals, errs, colors, labels = [], [], [], []
        for m in PAPER_METHODS:
            sub = st[(st['method'] == m) & (st['alpha'] == alpha)]
            if sub.empty:
                vals.append(0.0)
                errs.append(0.0)
            else:
                r = sub.iloc[0]
                scale = 100 if is_pct else 1
                vals.append(r['mean_l5'] * scale)
                errs.append(r['mean_l5_std'] * scale)
            colors.append(COLORS[m])
            labels.append(LABELS[m])

        x = np.arange(len(PAPER_METHODS))
        bars = ax.bar(x, vals, yerr=errs, capsize=3,
                      color=colors, width=0.6, alpha=0.85,
                      error_kw={'linewidth': 1.2})
        # bold border on SPA-M
        spa_idx = PAPER_METHODS.index('spa_m')
        bars[spa_idx].set_edgecolor('black')
        bars[spa_idx].set_linewidth(1.8)

        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
        ylabel = f'{METRIC_LABEL[ds]} (%)' if is_pct else METRIC_LABEL[ds]
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f'{DATASET_NAME[ds].split("(")[0].strip()}  α={alpha}',
                     fontweight='bold', fontsize=9)

        # annotate top value
        for bar, v, e in zip(bars, vals, errs):
            if v > 0:
                fmt = f'{v:.1f}' if is_pct else f'{v:.3f}'
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + e + (max(vals)*0.01),
                        fmt, ha='center', va='bottom', fontsize=6.5)

plt.tight_layout()
plt.savefig('../figures/all_datasets_bar_v2.pdf', bbox_inches='tight')
plt.show()

---
## 5  LaTeX Main Table

In [ ]:
# ── LaTeX Table 1 ─────────────────────────────────────────────────────────────
# Rows = methods, Cols = dataset × alpha, cell = MeanL5 ± std
# HetLoRA in italic, SPA-M bold

def fmt_cell(st, method, alpha, is_pct):
    sub = st[(st['method'] == method) & (st['alpha'] == alpha)]
    if sub.empty:
        return '—'
    r = sub.iloc[0]
    scale = 100 if is_pct else 1
    dp    = 1 if is_pct else 2
    mean  = r['mean_l5'] * scale
    std   = r['mean_l5_std'] * scale
    n     = int(r['n_seeds'])
    s = f'{mean:.{dp}f}\\tiny{{$\\pm${std:.{dp}f}}}'
    return s

# Find best per column to bold/underline
def best_in_col(ds, alpha):
    st = stats[ds]
    is_pct = ds in ('yelp', 'gsm8k')
    best_m, best_v = None, -1
    for m in PAPER_METHODS:
        sub = st[(st['method'] == m) & (st['alpha'] == alpha)]
        if sub.empty: continue
        v = sub.iloc[0]['mean_l5']
        if v > best_v:
            best_v, best_m = v, m
    return best_m

bests = {(ds, alpha): best_in_col(ds, alpha)
         for ds in DATASETS for alpha in ALPHAS}

# Build LaTeX
col_spec = 'l' + 'cc' * len(DATASETS)
header1  = ' & '.join(
    [' '] + [f'\\multicolumn{{2}}{{c}}{{{DATASET_NAME[ds].split("(")[0].strip()}}}'
             for ds in DATASETS])
header2  = ' & '.join(
    ['Method'] + [f'$\\alpha$=0.5 & $\\alpha$=0.1' for _ in DATASETS])

tex_lines = [
    '\\begin{table}[t]',
    '  \\centering',
    '  \\caption{Main results: Mean of Last 5 Rounds $\\pm$ std across seeds.',
    '    \\textbf{Bold} = best per column. \\textit{Italic} = external baselines.',
    '    SPA-M uses heterogeneous LoRA ranks with momentum aggregation.}',
    '  \\label{tab:main}',
    f'  \\begin{{tabular}}{{{col_spec}}}',
    '    \\toprule',
    f'    {header1} \\\\',
]
# cmidrule per dataset
cmid = ''
for i, ds in enumerate(DATASETS):
    start = 2 + i*2
    end   = start + 1
    cmid += f'\\cmidrule(lr){{{start}-{end}}} '
tex_lines.append(f'    {cmid.strip()} \\\\')
tex_lines.append(f'    {header2} \\\\')
tex_lines.append('    \\midrule')

for m in PAPER_METHODS:
    cells = []
    for ds in DATASETS:
        is_pct = ds in ('yelp', 'gsm8k')
        for alpha in ALPHAS:
            cell = fmt_cell(stats[ds], m, alpha, is_pct)
            if bests.get((ds, alpha)) == m:
                cell = f'\\textbf{{{cell}}}'
            cells.append(cell)
    label = LABELS.get(m, m)
    if m == 'spa_m':
        label = f'\\textbf{{{label}}}'
    elif m == 'hetlora':
        label = f'\\textit{{{label}}}'
    line = f"    {label} & {' & '.join(cells)} \\\\"
    tex_lines.append(line)

tex_lines += [
    '    \\bottomrule',
    '  \\end{tabular}',
    '\\end{table}',
]

latex_str = '\n'.join(tex_lines)
print(latex_str)

In [ ]:
# ── Save LaTeX to file ─────────────────────────────────────────────────────────
os.makedirs('../paper', exist_ok=True)
with open('../paper/table_main.tex', 'w') as f:
    f.write(latex_str)
print('Saved to ../paper/table_main.tex')

---
## 6  Missing Runs Checklist

In [ ]:
# ── Which (method × alpha × seed) runs are still missing? ─────────────────────
YELP_SEEDS  = [42, 43, 44, 45, 46]
GEN_SEEDS   = [42, 43, 44]
SEEDS_FOR   = {'yelp': YELP_SEEDS, 'gsm8k': GEN_SEEDS, 'alpaca': GEN_SEEDS}

missing = []
for ds in DATASETS:
    df = dfs[ds]
    for m in PAPER_METHODS:
        for alpha in ALPHAS:
            for seed in SEEDS_FOR[ds]:
                has = not df[
                    (df['method'] == m) &
                    (df['alpha']  == alpha) &
                    (df['seed']   == seed)
                ].empty
                if not has:
                    missing.append({'dataset': ds, 'method': m,
                                    'alpha': alpha, 'seed': seed})

if missing:
    miss_df = pd.DataFrame(missing)
    print(f'{len(missing)} missing runs:')
    display(miss_df.sort_values(['dataset','method','alpha','seed']))
else:
    print('All expected runs present!')